# ADK: Multi-Agent Collaboration via A2A Protocol (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_multi_agent_a2a_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_multi_agent_a2a_demo.ipynb)

This notebook shows how two ADK agents collaborate over the **A2A (Agent-to-Agent) Protocol** — one acting as a manager, the other as a remote specialist.

## Use Case
A legal review requires a **Manager Agent** (orchestrator) and a **Legal Specialist** (domain expert). Using A2A:
1.  **Connect**: The manager discovers the specialist via its agent card URL.
2.  **Delegate**: The manager sends a 'Contract Review' task to the specialist.
3.  **Collaborate**: They work together over several turns to produce a risk assessment.

### Release Notes
- [ADK v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0) — A2A lifespan parameter, new A2A-ADK extension
- [ADK v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0) — New RemoteA2aAgent implementation, A2A request interceptors

### Requirements
- `google-adk >= 1.28.0` and `a2a-sdk >= 0.3.25` installed.
- Gemini 3.1 Pro (Preview) access.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" "a2a-sdk>=0.3.25" google-genai nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'us-central1'  # @param {type:"string"}
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [PREREQUISITES] Define the Specialist Agent

We define a **Legal Specialist** agent that will be served as an A2A-capable endpoint. In production, this agent runs as a standalone service.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

# The Specialist Agent (Domain Expert)
legal_specialist = Agent(
    model="gemini-3.1-pro-preview",
    name="LegalSpecialist",
    instruction="You are a legal expert specializing in commercial contracts. Provide detailed risk analysis for any document provided."
)

print(f"Specialist agent '{legal_specialist.name}' defined.")

### 3. Core Feature: Serve the Specialist via A2A Protocol

`to_a2a()` turns any ADK agent into a Starlette HTTP app that speaks the A2A protocol. We run it as a background server so the manager can connect to it.

**Key pattern**: A2A is a server/client protocol — the specialist runs as a service, the manager connects remotely.

In [ ]:
import threading
import uvicorn
import time
from google.adk.a2a.utils.agent_to_a2a import to_a2a

# 1. Create the A2A server app from the specialist agent
A2A_PORT = 8765
specialist_app = to_a2a(
    agent=legal_specialist,
    host="0.0.0.0",
    port=A2A_PORT
)

# 2. Run the A2A server in a background thread
server_config = uvicorn.Config(specialist_app, host="0.0.0.0", port=A2A_PORT, log_level="warning")
server = uvicorn.Server(server_config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(2)  # Allow server to start

print(f"A2A Server running at http://localhost:{A2A_PORT}")
print(f"Agent Card available at: http://localhost:{A2A_PORT}/.well-known/agent-card.json")

### 4. Core Feature: Manager Connects via RemoteA2aAgent

The Manager uses `RemoteA2aAgent` as a sub-agent. It only needs the agent card URL — A2A handles capability discovery, the handshake, and multi-turn delegation.

In [ ]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

# 1. Connect to the specialist via A2A protocol (just needs the URL)
remote_specialist = RemoteA2aAgent(
    name="LegalSpecialist",
    agent_card=f"http://localhost:{A2A_PORT}/.well-known/agent-card.json",
    description="Remote legal specialist agent for contract review and risk analysis."
)

# 2. Define the Manager (Orchestrator) with the remote specialist as a sub-agent
manager_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="ProjectManager",
    instruction="""You are a project manager. When asked to review contracts or assess legal risks,
    delegate the task to the LegalSpecialist agent and summarize their findings for the user.""",
    sub_agents=[remote_specialist]
)

runner = Runner(
    agent=manager_agent,
    session_service=InMemorySessionService(),
    app_name="a2a_demo",
    auto_create_session=True
)

async def run_a2a_workflow():
    print("--- Starting A2A Multi-Agent Collaboration ---")
    prompt = "Analyze the liability clause in the draft contract doc_v1.pdf and provide a risk assessment."
    print(f"User: {prompt}\n")

    # Modern Runner Pattern (March 2026 Standard for Industrialized Apps)
    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    async for event in runner.run_async(
        user_id="partner_user",
        session_id="march_session",
        new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"{event.author}: {part.text}")
                if part.function_call:
                    print(f"[SYSTEM]: {event.author} delegating to '{part.function_call.name}'")

await run_a2a_workflow()

### 5. Things to remember or know
- **Server/client model**: The specialist runs as an HTTP service (via `to_a2a()`), the manager connects via `RemoteA2aAgent`. This enables cross-team, cross-cloud, and cross-framework agent collaboration.
- **Agent card discovery**: `RemoteA2aAgent` only needs the card URL (`/.well-known/agent-card.json`). Capability exchange and security are handled by the protocol. Note: the older `/.well-known/agent.json` endpoint is deprecated.
- **Works like local sub-agents**: `RemoteA2aAgent` plugs into ADK's `sub_agents` system, so routing and delegation work the same for local and remote agents.
- **Vertex AI auth**: Set `GOOGLE_GENAI_USE_VERTEXAI=TRUE`, `GOOGLE_CLOUD_PROJECT`, and `GOOGLE_CLOUD_LOCATION` **before** importing ADK. Without these, requests go to the AI Studio endpoint and fail with `API_KEY_INVALID`.
- **Lifespan management**: `to_a2a()` accepts a `lifespan` parameter for startup/shutdown hooks — useful for connection pooling in production.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.
- **Releases**: RemoteA2aAgent in [v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0). A2A lifespan and extension in [v1.28.0](https://github.com/google/adk-python/releases/tag/v1.28.0).